In [ ]:
import requests
import geopandas as gpd
from shapely.geometry import shape

# GeoAdmin-REST-Endpunkt
layer = "ch.swisstopo.swissboundaries3d-kanton-flaeche.fill"
base_url = "https://api3.geo.admin.ch/rest/services/api/MapServer"

# 1) Metadaten abrufen, um Feature-Anzahl / IDs zu prüfen
resp_meta = requests.get(f"{base_url}/{layer}")
resp_meta.raise_for_status()
meta = resp_meta.json()

print("Layer-Metadaten:", meta)

# 2) Features abrufen – hier alle Kantonsflächen (ACHTUNG: könnte viele sein)
# Optional: Mit Query Paramtern filtern, z. B. nur aktuelle Geometrien, layerDefs, etc.
params = {
    "geometryFormat": "geojson",
    "returnGeometry": "true",
    "sr": 4326  # WGS84, je nach Bedarf
}
resp = requests.get(f"{base_url}/{layer}/query", params=params)
resp.raise_for_status()

data = resp.json()

# 3) In GeoPandas laden
gdf = gpd.GeoDataFrame.from_features(data["features"])
print(gdf.head())

# 4) Nach Kanton SG filtern (BFS-Nummer = 17 laut swissBOUNDARIES3D)
# Prüfe zuerst, welches Attribut die Kantonsnummer hat – im Metadaten-PDF steht "KANTONSNUMMER". :contentReference[oaicite:4]{index=4}
gdf_sg = gdf[gdf["kantonsnummer"] == 17]  
print(gdf_sg)


HTTPError: 404 Client Error: Not Found for url: https://api3.geo.admin.ch/rest/services/api/MapServer/ch.swisstopo.swissboundaries3d-kanton-flaeche.fill/features?searchText=SG&geometryFormat=geojson&returnGeometry=true&sr=4326